In [1]:
pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd

# Load all CSV files
import pandas as pd

shipments = pd.read_csv("data/shipments.csv", sep=";")
routes = pd.read_csv("data/routes.csv", sep=";")
orders = pd.read_csv("data/orders.csv", sep=";")
carriers = pd.read_csv("data/carriers.csv", sep=";")
warehouses = pd.read_csv("data/warehouses.csv", sep=";")
slots = pd.read_csv("data/slots.csv", sep=";")

# Step 1: Join SHIPMENT with ROUTE
master = shipments.merge(
    routes,
    on="route_id",
    how="left"
)

# Step 2: Join with CARRIER
master = master.merge(
    carriers,
    left_on="assigned_carrier_id",
    right_on="carrier_id",
    how="left"
)

# Step 3: Join with WAREHOUSE
master = master.merge(
    warehouses,
    on="warehouse_id",
    how="left"
)

# Step 4: Join with SLOT
master = master.merge(
    slots,
    on="slot_id",
    how="left"
)

# Step 5: Join with ORDER
master = master.merge(
    orders,
    on="order_id",
    how="left"
)

# Step 6: Rename columns for clarity
master.rename(columns={
    "congestion_score_x": "route_congestion_score",
    "congestion_score_y": "warehouse_congestion_score",
    "ontime_percentage": "carrier_ontime_percentage",
    "avg_delay_minutes": "carrier_avg_delay_minutes"
}, inplace=True)

# Step 7: Save master dataset
master.to_csv("shipment_master_dataset.csv", index=False)

print("Master dataset created successfully")



In [ ]:
print(master.isnull().sum())


shipment_id             0
order_id                0
route_id_x              0
assigned_carrier_id     0
shipment_type           0
                       ..
pickup_location_id      0
delivery_location_id    0
promised_eta            0
order_status            0
price                   0
Length: 91, dtype: int64


In [ ]:
df = pd.read_csv("shipment_master_dataset.csv")

df.head()


,shipment_id,order_id,route_id_x,assigned_carrier_id,shipment_type,consolidation_allowed,consolidation_reason,planned_dispatch_time,actual_dispatch_time,planned_arrival_time,...,goods_type,material,plant,storage_location,goods_sensitivity,pickup_location_id,delivery_location_id,promised_eta,order_status,price
0,SHIP-0000001,ORD-0000001,R-BLR-HYD,C1-R-BLR-HYD,LTL,False,NaN,2026-01-02 13:00,2026-01-02 13:00,2026-01-02 21:00,...,Pharma,MAT-1739,BAN-PLANT,SL-11,Normal,WH-BAN-01,WH-HYD-01,2026-02-11 06:00,Confirmed,603053
1,SHIP-0000002,ORD-0000002,R-BLR-CHN,C3-R-BLR-CHN,LTL,True,NaN,2026-02-04 05:00,2026-02-04 05:00,2026-02-04 11:00,...,Auto Parts,MAT-7288,BAN-PLANT,SL-7,Fragile,WH-BAN-01,WH-CHE-01,2026-01-15 06:00,Confirmed,556157
2,SHIP-0000003,ORD-0000003,R-BLR-MUM,C2-R-BLR-MUM,FTL,True,NaN,2026-03-30 01:00,2026-03-30 01:00,2026-03-30 17:00,...,Auto Parts,MAT-9197,BAN-PLANT,SL-4,Temperature Controlled,WH-BAN-01,WH-MUM-01,2026-03-02 06:00,Confirmed,429755
3,SHIP-0000004,ORD-0000004,R-BLR-DEL,C1-R-BLR-DEL,FTL,False,NaN,2026-01-11 13:00,2026-01-11 13:00,2026-01-13 01:00,...,Steel,MAT-1912,BAN-PLANT,SL-15,Hazardous,WH-BAN-01,WH-DEL-01,2026-03-17 06:00,Confirmed,711695
4,SHIP-0000005,ORD-0000005,R-DEL-MUM,C2-R-DEL-MUM,FTL,False,NaN,2026-03-08 00:00,2026-03-08 00:00,2026-03-09 00:00,...,Auto Parts,MAT-8255,DEL-PLANT,SL-6,Normal,WH-DEL-01,WH-MUM-01,2026-04-02 06:00,Confirmed,687770


In [ ]:
df.shape

(24000, 91)

In [ ]:
master = pd.read_csv("shipment_master_dataset.csv")

print(master.shape)
print(master.columns)



(24000, 91)
Index(['shipment_id', 'order_id', 'route_id_x', 'assigned_carrier_id',
       'shipment_type', 'consolidation_allowed', 'consolidation_reason',
       'planned_dispatch_time', 'actual_dispatch_time', 'planned_arrival_time',
       'actual_arrival_time', 'shipment_status', 'warehouse_id_x', 'slot_id',
       'planned_freight_cost', 'actual_freight_cost', 'detention_cost',
       'delay_minutes', 'deviation_flag', 'calculated_delay_minutes',
       'root_cause', 'origin_city', 'destination_city', 'origin_warehouse_id',
       'destination_warehouse_id', 'distance_km',
       'standard_transit_time_hours', 'toll_cost_estimate',
       'fuel_cost_estimate', 'route_congestion_score', 'weather_risk_score',
       'historical_ontime_percentage', 'route_status',
       'route_congestion_score.1', 'carrier_id', 'carrier_name', 'route_id_y',
       'volume_commitment_percentage', 'agreed_cost_per_1000_shipments',
       'daily_capacity', 'monthly_capacity', 'current_allocated_shipmen

In [ ]:
import pandas as pd
import numpy as np

# =========================
# FIX DATETIME COLUMNS
# =========================

datetime_cols = [

    'planned_dispatch_time',
    'actual_dispatch_time',
    'planned_arrival_time',
    'actual_arrival_time',

    'actual_checkin_time',
    'actual_checkout_time',

    'slot_start_time',
    'slot_end_time',

    'order_date',
    'delivery_due_date',
    'promised_eta'
]

for col in datetime_cols:

    if col in master.columns:

        master[col] = pd.to_datetime(
            master[col],
            errors='coerce'
        )


# =========================
# CREATE STRONG FEATURES
# =========================

master['planned_duration_minutes'] = (
    master['planned_arrival_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60


master['actual_duration_minutes'] = (
    master['actual_arrival_time'] -
    master['actual_dispatch_time']
).dt.total_seconds() / 60


master['dispatch_delay_minutes'] = (
    master['actual_dispatch_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60


master['waiting_time_minutes'] = (
    master['actual_checkin_time'] -
    master['slot_start_time']
).dt.total_seconds() / 60


master['duration_deviation_minutes'] = (
    master['actual_duration_minutes'] -
    master['planned_duration_minutes']
)


master['dispatch_hour'] = master['planned_dispatch_time'].dt.hour

master['dispatch_dayofweek'] = master['planned_dispatch_time'].dt.dayofweek

master['is_weekend'] = (
    master['dispatch_dayofweek'] >= 5
).astype(int)


# =========================
# FIX NUMERIC COLUMNS
# =========================

numeric_cols = master.select_dtypes(include='object').columns

for col in numeric_cols:

    master[col] = pd.to_numeric(
        master[col].astype(str).str.replace(',', '.'),
        errors='ignore'
    )


# =========================
# VERIFY FEATURES CREATED
# =========================

print(master[['planned_duration_minutes',
              'dispatch_delay_minutes',
              'waiting_time_minutes']].head())

   planned_duration_minutes  dispatch_delay_minutes  waiting_time_minutes
0                     480.0                     0.0                   NaN
1                     360.0                     0.0                   NaN
2                     960.0                     0.0                   NaN
3                    2160.0                     0.0                   NaN
4                    1440.0                     0.0                   NaN


C:\Users\Abcom\AppData\Local\Temp\ipykernel_11940\1674825576.py:87: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  master[col] = pd.to_numeric(
C:\Users\Abcom\AppData\Local\Temp\ipykernel_11940\1674825576.py:87: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  master[col] = pd.to_numeric(
C:\Users\Abcom\AppData\Local\Temp\ipykernel_11940\1674825576.py:87: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  master[col] = pd.to_numeric(
C:\Users\Abcom\AppData\Local\Temp\ipykernel_11940\1674825576.py:87: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly

In [ ]:
# Dispatch delay
master['dispatch_delay_minutes'] = (
    master['actual_dispatch_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60


# Planned duration
master['planned_duration_minutes'] = (
    master['planned_arrival_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60


# Waiting time
master['waiting_time_minutes'] = (
    master['actual_checkin_time'] -
    master['slot_start_time']
).dt.total_seconds() / 60


# Time intelligence features
master['dispatch_hour'] = master['planned_dispatch_time'].dt.hour

master['dispatch_dayofweek'] = master['planned_dispatch_time'].dt.dayofweek

master['is_weekend'] = (
    master['dispatch_dayofweek'] >= 5
).astype(int)

In [ ]:
master['planned_duration_minutes'] = (
    pd.to_datetime(master['planned_arrival_time']) -
    pd.to_datetime(master['planned_dispatch_time'])
).dt.total_seconds() / 60

master['dispatch_delay_minutes'] = (
    pd.to_datetime(master['actual_dispatch_time']) -
    pd.to_datetime(master['planned_dispatch_time'])
).dt.total_seconds() / 60

master['carrier_capacity_utilization'] = (
    master['current_allocated_shipments'] /
    master['daily_capacity']
)

master['weight_per_pallet'] = (
    master['total_weight_kg'] /
    master['pallet_count']
)

master['volume_per_pallet'] = (
    master['total_volume_cbm'] /
    master['pallet_count']
)

In [ ]:
master['planned_dispatch_time'] = pd.to_datetime(master['planned_dispatch_time'])

master['dispatch_hour'] = master['planned_dispatch_time'].dt.hour
master['dispatch_dayofweek'] = master['planned_dispatch_time'].dt.dayofweek
master['is_weekend'] = master['dispatch_dayofweek'].isin([5,6]).astype(int)

In [ ]:
master['carrier_utilization'] = (
    master['current_allocated_shipments'] /
    master['daily_capacity']
)

master['route_risk_score'] = (
    master['route_congestion_score'] +
    master['weather_risk_score']
)

master['cost_deviation'] = (
    master['actual_freight_cost'] -
    master['planned_freight_cost']
)

In [ ]:
features = [

    'planned_duration_minutes',
    'dispatch_delay_minutes',
    'waiting_time_minutes',

    'carrier_utilization',

    'carrier_avg_delay_minutes',
    'carrier_ontime_percentage',

    'route_congestion_score',
    'weather_risk_score',
    'route_risk_score',

    'warehouse_congestion_score',

    'distance_km',

    'total_weight_kg',
    'total_volume_cbm',
    'pallet_count',

    'dispatch_hour',
    'dispatch_dayofweek',
    'is_weekend',

    'cost_deviation'
]

In [ ]:
master['carrier_capacity_utilization'] = (
    master['current_allocated_shipments'] /
    master['daily_capacity']
)

master['weight_per_pallet'] = (
    master['total_weight_kg'] /
    master['pallet_count']
)

master['volume_per_pallet'] = (
    master['total_volume_cbm'] /
    master['pallet_count']
)

In [ ]:
master['planned_dispatch_time'] = pd.to_datetime(master['planned_dispatch_time'])
master['actual_dispatch_time'] = pd.to_datetime(master['actual_dispatch_time'])
master['planned_arrival_time'] = pd.to_datetime(master['planned_arrival_time'])
master['actual_arrival_time'] = pd.to_datetime(master['actual_arrival_time'])

In [ ]:
master['planned_duration_minutes'] = (
    master['planned_arrival_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60

master['dispatch_delay_minutes'] = (
    master['actual_dispatch_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60

In [ ]:
master['calculated_delay_minutes'] = (
    master['actual_arrival_time'] -
    master['planned_arrival_time']
).dt.total_seconds() / 60

In [ ]:
master = master[master['calculated_delay_minutes'].notna()]

In [ ]:
# Planned duration
master['planned_duration_minutes'] = (
    master['planned_arrival_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60

# Dispatch delay
master['dispatch_delay_minutes'] = (
    master['actual_dispatch_time'] -
    master['planned_dispatch_time']
).dt.total_seconds() / 60

# Waiting time at warehouse
master['waiting_time_minutes'] = (
    master['actual_checkin_time'] -
    master['slot_start_time']
).dt.total_seconds() / 60

# Dispatch hour
master['dispatch_hour'] = master['planned_dispatch_time'].dt.hour

# Day of week
master['dispatch_dayofweek'] = master['planned_dispatch_time'].dt.dayofweek

# Weekend flag
master['is_weekend'] = master['dispatch_dayofweek'].isin([5,6]).astype(int)

In [ ]:
for col in features:

    if col in master.columns:

        master[col] = (
            master[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
        )

        master[col] = pd.to_numeric(master[col], errors='coerce')

In [ ]:
X = master[features].fillna(master[features].median())

y = master['calculated_delay_minutes']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

reg_model = RandomForestRegressor(
    n_estimators=800,
    max_depth=18,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

reg_model.fit(X_train, y_train)

,n_estimators,800
,criterion,'squared_error'
,max_depth,18
,min_samples_split,4
,min_samples_leaf,2
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
from sklearn.metrics import mean_squared_error

y_pred = reg_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

print("Final RMSE:", rmse, "minutes")

Final RMSE: 70.7332562295848 minutes


In [ ]:
print("Rows:", len(master))
print(master['calculated_delay_minutes'].describe())
print(master['calculated_delay_minutes'].skew())

Rows: 24000
count    24000.000000
mean        72.665417
std         95.281851
min        -90.000000
25%        -10.000000
50%         72.000000
75%        154.000000
max        240.000000
Name: calculated_delay_minutes, dtype: float64
0.025366126623930455


In [ ]:
reg_model.predict(X_train[:5])


array([ 71.52358746, 152.88906801,  36.02373577,  37.81168015,
       144.49080484])

In [ ]:
import pickle

# Save model
with open("delay_model.pkl", "wb") as f:
    pickle.dump(reg_model, f)

# Save feature columns (VERY IMPORTANT)
with open("model_features.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("Model saved successfully")

Model saved successfully
